# LLM Benchmark Framework Guide

This notebook provides a comprehensive guide for the enhanced LLM Benchmark Framework, showcasing all the latest features including:

- **Enhanced Metrics System** with factory pattern and comprehensive validation
- **Dataset Loading** from multiple formats (JSON, CSV, YAML, TXT)
- **Model Factory** for easy model creation and management
- **Enhanced Visualizations** with 6 different chart types
- **Interactive HTML Dashboard** with detailed analytics
- **Comprehensive Statistics** and analysis capabilities

## Step 1: Import Enhanced Modules

Import all the enhanced modules with factory patterns and improved functionality.

In [1]:
import asyncio
import os
from datetime import datetime

# Enhanced imports with factory patterns
from models import ModelFactory
from metrics import MetricFactory, EvaluatorFactory
from dataset import DatasetLoader, Dataset
from benchmark.runner import BenchmarkRunner
from visualizations.evaluation_visualizer import EvaluationVisualizer
from dashboard import generate_html_dashboard
from metrics.responses import BenchmarkResult

print("All modules imported successfully!")
print(f"Available metrics: {', '.join(MetricFactory.list_available_metrics())}")
print(f"Available model types: {', '.join(ModelFactory.list_available_models())}")

/home/benedikt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:149: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


All modules imported successfully!
Available metrics: relevance, hallucinations, fairness, robustness, bias, toxicity
Available model types: openai, ollama


## Step 2: Load Dataset from File

Demonstrate the enhanced dataset loading capabilities with rich metadata support.

In [2]:
# Load the sample dataset we created
dataset = DatasetLoader.load_from_file("sample_questions.json")

print(f"Dataset loaded: {dataset.name}")
print(f"Description: {dataset.description}")
print(f"Number of questions: {len(dataset)}")

# Display dataset statistics
stats = dataset.get_statistics()
print(f"Dataset Statistics:")
print(f"  • Languages: {list(stats['languages'].keys())}")
print(f"  • Average text length: {stats['average_text_length']:.1f} characters")
print(f"  • Questions with context: {stats['has_context']}")
print(f"  • Questions with instructions: {stats['has_instructions']}")
print(f"  • Questions with expected answers: {stats['has_expected_answers']}")

# Show a sample question with rich metadata
sample_question = dataset.questions[1]  # Get the quantum computing question
print(f"\nSample Question (ID: {sample_question.id}):")
print(f"  Text: {sample_question.text}")
print(f"  Context: {sample_question.context}")
print(f"  Instructions: {sample_question.instructions}")
print(f"  Domain: {sample_question.metadata.get('domain')} - {sample_question.metadata.get('type')}")

Dataset loaded: LLM Benchmark Sample Dataset
Description: A comprehensive sample dataset for demonstrating LLM benchmarking capabilities across various domains and difficulty levels
Number of questions: 10
Dataset Statistics:
  • Languages: ['en']
  • Average text length: 64.2 characters
  • Questions with context: 10
  • Questions with instructions: 10
  • Questions with expected answers: 10

Sample Question (ID: q002):
  Text: Explain the concept of quantum computing in simple terms.
  Context: You are explaining to someone with basic computer knowledge but no physics background.
  Instructions: Use analogies and simple language to make the concept accessible.
  Domain: technology - conceptual_explanation


## Step 3: Enhanced Model Setup

Use the ModelFactory to create models with enhanced configuration and validation.

In [3]:
# Configure models optimized for slower hardware
print("Optimizing configuration for local hardware...")

# Use smaller, faster model for testing with longer timeout
test_model = ModelFactory.create_ollama_model(
    model_name="gemma3:1b",  # Much smaller and faster than llama3.2
    base_url="http://localhost:11434",
    temperature=0.7,
    timeout=300  # 5 minutes timeout for slow hardware
)

print(f"Test model created: {test_model.config.model_name} (optimized for speed)")

# Use single evaluator model to reduce computational load
evaluator_models = [
    ModelFactory.create_ollama_model(
        model_name="gemma3:1b",  # Fast and lightweight
        temperature=0.3,
        timeout=300
    )
]

evaluator = EvaluatorFactory.create_evaluator(evaluator_models)
print(f" Created single-model evaluator (faster for local testing)")

# Display model information
for i, model in enumerate(evaluator_models):
    model_info = model.get_model_info()
    print(f"  {i+1}. {model.config.model_name} - Lightweight model for local testing")

Optimizing configuration for local hardware...
Test model created: gemma3:1b (optimized for speed)
 Created single-model evaluator (faster for local testing)
  1. gemma3:1b - Lightweight model for local testing


## Step 4: Enhanced Metrics System

Showcase the new MetricFactory with comprehensive metrics and validation.

In [4]:
# List all available metrics with descriptions
available_metrics = MetricFactory.list_available_metrics()
print("Available Metrics:")
for metric_name in available_metrics:
    metric = MetricFactory.create_metric(metric_name)
    print(f"  • {metric_name}: {metric.description}")

# For faster testing on slower hardware, use fewer metrics
print("Optimizing for local testing - using subset of metrics")
selected_metrics = MetricFactory.create_metrics_by_names(["relevance", "hallucinations"])
print(f"Using {len(selected_metrics)} metrics for faster evaluation")

# Use the optimized metrics
metrics = selected_metrics

Available Metrics:
  • relevance: Measures how well the response addresses the prompt's requirements and stays on topic
  • hallucinations: Measures the extent to which the response contains fabricated or incorrect information
  • fairness: Measures how fairly the response treats different groups and perspectives
  • robustness: Measures how well the response handles edge cases, ambiguity, and challenging inputs
  • bias: Measures the presence of demographic, cultural, or ideological bias in the response
  • toxicity: Measures the presence of harmful, toxic, or inappropriate content in the response
Optimizing for local testing - using subset of metrics
Using 2 metrics for faster evaluation


## Step 5: Run Enhanced Benchmark

Execute the benchmark with enhanced error handling and comprehensive logging.

In [5]:
# Create benchmark runner with optimized metrics
runner = BenchmarkRunner(evaluator, metrics)

async def run_comprehensive_benchmark():
    print("Starting optimized benchmark for local testing...")

    # Use smaller dataset for faster testing on slower hardware
    test_dataset = dataset.sample(3, random_seed=42)  # Only 3 questions for testing
    print(f"Dataset: {test_dataset.name} ({len(test_dataset)} questions - optimized)")
    print(f"Metrics: {len(metrics)} evaluation criteria")
    print(f"Evaluators: {len(evaluator_models)} models")
    print("Using reduced dataset and metrics for local hardware")
    
    try:
        # Run the benchmark with the smaller dataset
        benchmark_result = await runner.run_benchmark(test_model, test_dataset)
        print("Benchmark completed successfully!")
        return benchmark_result
    except Exception as e:
        print(f"Benchmark failed: {e}")
        print("Try checking if Ollama is running: 'ollama serve'")
        raise

# Execute the benchmark
benchmark_result: BenchmarkResult = await run_comprehensive_benchmark()

results_file = f"results/benchmark_results.json"
os.makedirs("results", exist_ok=True)
benchmark_result.save_to_json_file(results_file)
print(f"Results saved to: {results_file}")

[ROCKET] Starting optimized benchmark for local testing...
Dataset: LLM Benchmark Sample Dataset (Sample 3) (3 questions - optimized)
Metrics: 2 evaluation criteria
Evaluators: 1 models
Using reduced dataset and metrics for local hardware
Benchmark completed successfully!
[DISK] Results saved to: results/benchmark_results_20250629_230850.json


## Step 6: Comprehensive Results Analysis

Analyze the results with enhanced statistics and insights.

In [6]:
# Get comprehensive summary statistics
summary = benchmark_result.get_summary_statistics()

print("BENCHMARK RESULTS SUMMARY")
print("=" * 50)
print(f"Model: {summary['model_name']}")
print(f"Dataset: {dataset.name}")
print(f"Questions: {summary['num_prompts']}")
print(f"Metrics: {summary['num_metrics']}")
print(f"Overall Average: {summary['overall_average']:.2f}/10")

print(f"\n[CHART] Score Distribution:")
if 'score_distribution' in summary:
    dist = summary['score_distribution']
    print(f"  • Range: {dist['min']:.1f} - {dist['max']:.1f}")
    print(f"  • Median: {dist['median']:.1f}")
    print(f"  • Standard Deviation: {dist['std_dev']:.2f}")

print(f"Average Scores by Metric:")
for metric, score in summary['average_scores'].items():
    # Add emoji based on score
    if score >= 8:
        emoji = "🟢"
    elif score >= 6:
        emoji = "🟡"
    elif score >= 4:
        emoji = "🟠"
    else:
        emoji = "🔴"
    print(f"  {emoji} {metric}: {score:.2f}/10")

# Show detailed results for a few questions
print(f"\n[SEARCH] Sample Question Results:")
for i, evaluation in enumerate(benchmark_result.prompt_evaluations[:3]):
    print(f"\n  Question {i+1}: {evaluation.prompt[:60]}...")
    print(f"  Response: {evaluation.response[:100]}...")
    for eval_result in evaluation.evaluations:
        print(f"    • {eval_result.metric_name}: {eval_result.score:.1f}/10")

BENCHMARK RESULTS SUMMARY
Model: gemma3:1b
Dataset: LLM Benchmark Sample Dataset
Questions: 3
Metrics: 2
Overall Average: 6.50/10

[CHART] Score Distribution:
  • Range: 1.0 - 10.0
  • Median: 8.0
  • Standard Deviation: 3.10
Average Scores by Metric:
  🟢 relevance: 9.00/10
  🟠 hallucinations: 4.00/10

[SEARCH] Sample Question Results:

  Question 1: Context: You are explaining to someone with basic computer k...
  Response: Okay, let’s tackle quantum computing. Think of it this way: regular computers, like the one you’re u...
    • relevance: 8.0/10
    • hallucinations: 4.0/10

  Question 2: Context: You are answering a basic geography question.

Inst...
  Response: Paris....
    • relevance: 9.0/10
    • hallucinations: 1.0/10

  Question 3: Context: You are helping someone learn programming concepts....
  Response: Okay, let's craft a Python function to calculate the Fibonacci sequence.  Here's the code, thoroughl...
    • relevance: 10.0/10
    • hallucinations: 7.0/10


## Step 7: Enhanced Static Visualizations

Generate improved visualizations with better insights and professional styling.

In [3]:
results_file = "results/benchmark_results_20250629_230850.json"
benchmark_result = BenchmarkResult.load_from_json_file(results_file)

# Initialize the enhanced visualizer
visualizer = EvaluationVisualizer(results_dir="results")

print("Generating enhanced static visualizations...")

# 1. Benchmark Results (Bar Chart with Individual Scores)
print("Creating benchmark results chart...")
figure = visualizer.plot_benchmark_results(benchmark_result, f"Benchmark Results - {dataset.name}")
visualizer.save_plot(figure, "benchmark_results.png")

# 2. Performance Radar Chart (NEW)
print("Creating performance radar chart...")
figure = visualizer.plot_radar_chart(benchmark_result)
visualizer.save_plot(figure, "performance_radar.png")

# 3. Per-Question Heatmaps
print("Creating per-question heatmaps...")
figure = visualizer.plot_per_question_scores(benchmark_result)
visualizer.save_plot(figure, "per_question_scores.png")

# 4. Metric Correlation Matrix (NEW)
print("Creating metric correlation matrix...")
figure = visualizer.plot_metric_correlation_matrix(benchmark_result)
visualizer.save_plot(figure, "metric_correlations.png")

# 5. Question Difficulty Analysis (NEW)
print("Creating question difficulty analysis...")
figure = visualizer.plot_question_difficulty_analysis(benchmark_result)
visualizer.save_plot(figure, "question_difficulty.png")

# 6. Evaluator Agreement Analysis (NEW)
print("Creating evaluator agreement analysis...")
figure = visualizer.plot_evaluator_agreement(benchmark_result)
visualizer.save_plot(figure, "evaluator_agreement.png")

print("All static visualizations generated and saved to 'results/' directory")

Generating enhanced static visualizations...
Creating benchmark results chart...
Creating performance radar chart...
Creating per-question heatmaps...
Creating metric correlation matrix...
Creating question difficulty analysis...
Creating evaluator agreement analysis...
All static visualizations generated and saved to 'results/' directory


## Step 8: Interactive HTML Dashboard (NEW)

Generate a comprehensive interactive HTML dashboard with detailed analytics.

In [4]:
print("Generating interactive HTML dashboard...")

# Generate the comprehensive HTML dashboard
dashboard_path = generate_html_dashboard(benchmark_result, "results/dashboard.html")

print(f" Interactive dashboard generated: {dashboard_path}")
print("\nDashboard Features:")
print("Overview: Key metrics, radar charts, performance summaries")
print("Metrics Analysis: Correlation matrices, detailed statistics")
print("Questions Analysis: Difficulty analysis, per-question breakdowns")
print("Evaluators Analysis: Performance comparison, agreement analysis")
print(" Detailed View: Complete raw data with color-coded scores")
print("\nFeatures:")
print("  • Interactive charts with Chart.js and D3.js")
print("  • Responsive design for mobile compatibility")
print("  • Detailed question modals with full context")
print("  • Professional styling with Tailwind CSS")
print("  • Standalone HTML (works offline, no server needed)")

# Display the path for easy access
import os
full_path = os.path.abspath(dashboard_path)
print(f"\nOpen dashboard: file://{full_path}")
print(f"Or run: open {dashboard_path}")

Generating interactive HTML dashboard...
 Interactive dashboard generated: results/dashboard.html

Dashboard Features:
Overview: Key metrics, radar charts, performance summaries
Metrics Analysis: Correlation matrices, detailed statistics
Questions Analysis: Difficulty analysis, per-question breakdowns
Evaluators Analysis: Performance comparison, agreement analysis
 Detailed View: Complete raw data with color-coded scores

Features:
  • Interactive charts with Chart.js and D3.js
  • Responsive design for mobile compatibility
  • Detailed question modals with full context
  • Professional styling with Tailwind CSS
  • Standalone HTML (works offline, no server needed)

Open dashboard: file:///home/benedikt/uni/msi/teamprojekt/dev/llm_benchmark/results/dashboard.html
Or run: open results/dashboard.html
